In [1]:
import numpy as np

class LSTM:

    def __init__(self, input_size, hidden_size, output_size):

        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size

        # Forget Gate
        self.Wf = np.random.randn(hidden_size, hidden_size + input_size) * 0.01
        self.bf = np.zeros((hidden_size, 1))

        # Input Gate
        self.Wi = np.random.randn(hidden_size, hidden_size + input_size) * 0.01
        self.bi = np.zeros((hidden_size, 1))

        # Candidate Cell
        self.Wc = np.random.randn(hidden_size, hidden_size + input_size) * 0.01
        self.bc = np.zeros((hidden_size, 1))

        # Output Gate
        self.Wo = np.random.randn(hidden_size, hidden_size + input_size) * 0.01
        self.bo = np.zeros((hidden_size, 1))

        # Output Layer
        self.Wy = np.random.randn(output_size, hidden_size) * 0.01
        self.by = np.zeros((output_size, 1))

    
    # ACTIVATIONS


    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def dsigmoid(self, x):
        return x * (1 - x)

    def tanh(self, x):
        return np.tanh(x)

    def dtanh(self, x):
        return 1 - x**2

    def softmax(self, x):
        exp_x = np.exp(x - np.max(x))
        return exp_x / np.sum(exp_x)

   
    # FORWARD PASS
   
    def forward(self, X):

        self.cache = []

        h_prev = np.zeros((self.hidden_size, 1))
        c_prev = np.zeros((self.hidden_size, 1))

        for t in range(len(X)):

            x = X[t].reshape(-1, 1)

            concat = np.vstack((h_prev, x))

            f = self.sigmoid(self.Wf @ concat + self.bf)

            i = self.sigmoid(self.Wi @ concat + self.bi)

            c_bar = self.tanh(self.Wc @ concat + self.bc)

            c = f * c_prev + i * c_bar

            o = self.sigmoid(self.Wo @ concat + self.bo)

            h = o * self.tanh(c)

            self.cache.append(
                (concat, f, i, c_bar, c, o, h, c_prev)
            )

            h_prev = h
            c_prev = c

        y = self.softmax(self.Wy @ h + self.by)

        return y

   
    # LOSS
    
    def compute_loss(self, y_pred, y_true):

        return -np.sum(y_true * np.log(y_pred + 1e-8))

   
    # BACKWARD PASS
   
    def backward(self, X, y_true, y_pred, lr=0.01):

        dWy = np.zeros_like(self.Wy)
        dby = np.zeros_like(self.by)

        dWf = np.zeros_like(self.Wf)
        dWi = np.zeros_like(self.Wi)
        dWc = np.zeros_like(self.Wc)
        dWo = np.zeros_like(self.Wo)

        dbf = np.zeros_like(self.bf)
        dbi = np.zeros_like(self.bi)
        dbc = np.zeros_like(self.bc)
        dbo = np.zeros_like(self.bo)

        dy = y_pred - y_true

        dWy += dy @ self.cache[-1][6].T
        dby += dy

        dh_next = self.Wy.T @ dy
        dc_next = np.zeros((self.hidden_size, 1))

        for t in reversed(range(len(self.cache))):

            concat, f, i, c_bar, c, o, h, c_prev = self.cache[t]

            dh = dh_next

            do = dh * np.tanh(c)
            do_raw = do * self.dsigmoid(o)

            dc = dh * o * self.dtanh(np.tanh(c))
            dc += dc_next

            df = dc * c_prev
            df_raw = df * self.dsigmoid(f)

            di = dc * c_bar
            di_raw = di * self.dsigmoid(i)

            dc_bar = dc * i
            dc_bar_raw = dc_bar * self.dtanh(c_bar)

            dWf += df_raw @ concat.T
            dWi += di_raw @ concat.T
            dWc += dc_bar_raw @ concat.T
            dWo += do_raw @ concat.T

            dbf += df_raw
            dbi += di_raw
            dbc += dc_bar_raw
            dbo += do_raw

            dconcat = (
                self.Wf.T @ df_raw +
                self.Wi.T @ di_raw +
                self.Wc.T @ dc_bar_raw +
                self.Wo.T @ do_raw
            )

            dh_next = dconcat[:self.hidden_size, :]
            dc_next = dc * f

        # Gradient Descent

        self.Wf -= lr * dWf
        self.Wi -= lr * dWi
        self.Wc -= lr * dWc
        self.Wo -= lr * dWo

        self.bf -= lr * dbf
        self.bi -= lr * dbi
        self.bc -= lr * dbc
        self.bo -= lr * dbo

        self.Wy -= lr * dWy
        self.by -= lr * dby

   
    # TRAINING
   

    def fit(self, X_train, y_train,
            epochs=500, lr=0.01):

        for epoch in range(epochs):

            total_loss = 0

            for X, y in zip(X_train, y_train):

                y_pred = self.forward(X)

                loss = self.compute_loss(y_pred, y)

                self.backward(X, y, y_pred, lr)

                total_loss += loss

            if epoch % 50 == 0:

                print(
                    f"Epoch {epoch} | Loss: {total_loss:.4f}"
                )

    
    # PREDICT
   

    def predict(self, X):

        y = self.forward(X)

        return np.argmax(y)


In [2]:
# EXAMPLE DATA

X_train = [
    np.array([[1], [0], [1]]),
    np.array([[0], [1], [0]]),
    np.array([[1], [1], [1]]),
    np.array([[0], [0], [0]])
]

y_train = [
    np.array([[1], [0]]),
    np.array([[0], [1]]),
    np.array([[1], [0]]),
    np.array([[0], [1]])
]

model = LSTM(
    input_size=1,
    hidden_size=8,
    output_size=2
)

model.fit(
    X_train,
    y_train,
    epochs=500,
    lr=0.01
)

print("\nPrediction:")
print(model.predict(np.array([[1], [0], [1]])))

Epoch 0 | Loss: 2.7828
Epoch 50 | Loss: 2.7826
Epoch 100 | Loss: 2.7824
Epoch 150 | Loss: 2.7823
Epoch 200 | Loss: 2.7820
Epoch 250 | Loss: 2.7816
Epoch 300 | Loss: 2.7810
Epoch 350 | Loss: 2.7800
Epoch 400 | Loss: 2.7786
Epoch 450 | Loss: 2.7764

Prediction:
1
